# Stage B — Bibliometric and trend analysis
## Keyword distribution, publication trend, study types, journals

**AI adoption in urban planning governance: a systematic review**
Lartey & Law (2025), *Landscape and Urban Planning* 258, 105337

---

Section 4.2 of the paper. Four descriptive views of the corpus, assembled into the
paper's Figure 3, plus the application-area and discipline classification from
Section 4.4 (Figure 5a–b).

Every panel is computed from `corpus.csv`. The paper's published counts are held
alongside as comparison targets so you can see how far your corpus tracks it — they
are never used as figure input.

**Reads** `data/corpus.csv` (written by notebook 01)
**Writes** `outputs/figures/figure3_bibliometric.png`, `figure5ab_areas_disciplines.png`,
and the underlying tables to `outputs/tables/`.

In [ ]:
from __future__ import annotations

import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def _repo_relative(p):
    """Resolve whether the kernel started in notebooks/ or the repository root."""
    p = Path(p)
    if p.exists():
        return p
    alt = Path(str(p).replace("../", "", 1))
    return alt if (alt.exists() or Path("notebooks").is_dir()) else p


CONFIG = {"data_dir": "../data", "output_dir": "../outputs", "seed": 42, "dpi": 200}
SEED = CONFIG["seed"]
np.random.seed(SEED)

DATA_DIR = _repo_relative(CONFIG["data_dir"])
OUT_DIR = _repo_relative(CONFIG["output_dir"])
FIG_DIR, TAB_DIR = OUT_DIR / "figures", OUT_DIR / "tables"
for d in (FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": CONFIG["dpi"],
                     "savefig.bbox": "tight", "font.size": 9,
                     "axes.titlesize": 10, "axes.titleweight": "bold"})

PALETTE = ["#9ecae1", "#6baed6", "#4292c6", "#2171b5", "#08519c"]

CORPUS_PATH = DATA_DIR / "corpus.csv"
if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"{CORPUS_PATH.resolve()} not found.\n"
        f"Run 01_corpus_assembly.ipynb first - it writes this file.")

corpus = pd.read_csv(CORPUS_PATH)
corpus["Year"] = pd.to_numeric(corpus["Year"], errors="coerce")
corpus = corpus.dropna(subset=["Year"])
corpus["Year"] = corpus["Year"].astype(int)

IS_DEMO = corpus["SourceFile"].astype(str).eq("SYNTHETIC").any() \
    if "SourceFile" in corpus.columns else False


def stamp(fig, demo=None, text="DEMO DATA\nNOT PAPER RESULTS"):
    """Mark synthetic output. `demo` defaults to the corpus, but a figure built from
    another source (e.g. the VOSviewer network) passes its own flag."""
    if (IS_DEMO if demo is None else demo):
        fig.text(0.5, 0.5, text, fontsize=42, color="grey", alpha=0.13,
                 ha="center", va="center", rotation=30, zorder=1000, weight="bold")
    return fig


def finish(fig, name, demo=None, stamp_text="DEMO DATA\nNOT PAPER RESULTS"):
    stamp(fig, demo=demo, text=stamp_text)
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path)
    plt.show()
    print(f"  saved: {path}")


print(f"Corpus: {len(corpus):,} records, {corpus['Year'].min()}-{corpus['Year'].max()}")
print(f"Paper for comparison: 588 records across 83 countries")
if IS_DEMO:
    print("\n" + "!" * 66)
    print("This corpus is the SYNTHETIC demo set. Figures will be stamped.")
    print("!" * 66)

In [ ]:
# ---- Published values, kept ONLY as comparison targets ----
PAPER_STUDY_TYPES = {"Literature review": 283, "Systematic review": 180,
                     "Non-RCT observational study": 67, "Meta-analysis": 58}
PAPER_JOURNAL_SHARE = {"Sustainability": 19, "arXiv": 12, "Cities": 10,
                       "Journal of The American Planning Association": 10, "Other": 50}
PAPER_KEYWORD_SHARE = {"AI": 69, "decision-making": 22, "urban planning": 8,
                       "city planning": 1, "policy formulation": 1}
PAPER_APPLICATION_COUNTS = {"Community and Social Services": 278,
                            "Environmental Planning and Sustainability": 148,
                            "Smart Cities and Infrastructure Development": 69,
                            "Land Use and Zoning": 58,
                            "Transportation Planning and Management": 35}
PAPER_DISCIPLINE_COUNTS = {"Public Administration": 249, "Social Sciences": 234,
                           "Other": 40, "Engineering": 31, "Computer Science": 23,
                           "Urban Studies/Planning": 7, "Law & Regulation": 2,
                           "Economics": 1}

EXTRACTION_KEYWORDS = {
    "AI":                 ["artificial intelligence", " ai ", "machine learning",
                           "deep learning"],
    "decision-making":    ["decision-making", "decision making"],
    "urban planning":     ["urban planning"],
    "city planning":      ["city planning"],
    "policy formulation": ["policy formulation", "policymaking", "policy making"],
}

APPLICATION_AREAS = {
    "Community and Social Services": ["community", "social service", "public service",
                                      "citizen", "wellbeing", "well-being", "equity"],
    "Environmental Planning and Sustainability": ["environment", "sustainab", "climate",
                                                  "green", "ecolog", "emission"],
    "Smart Cities and Infrastructure Development": ["smart city", "smart cities",
                                                    "infrastructure", "iot", "digital twin"],
    "Land Use and Zoning": ["land use", "land-use", "zoning", "urban form"],
    "Transportation Planning and Management": ["transport", "mobility", "traffic",
                                               "autonomous vehicle", "logistics"],
}

DISCIPLINES = {
    "Public Administration": ["public administration", "governance", "policy",
                              "government", "public sector"],
    "Social Sciences":       ["social", "society", "sociolog", "behaviour", "behavior"],
    "Urban Studies/Planning": ["urban studies", "urban planning", "planning", "urbanism"],
    "Computer Science":      ["computer science", "algorithm", "computing", "software"],
    "Engineering":           ["engineering", "control system", "optimi"],
    "Law & Regulation":      ["law", "legal", "regulation", "compliance"],
    "Economics":             ["econom", "market", "cost-benefit"],
}


def text_blob(df, fields=("Title", "Abstract", "Keyword")):
    b = pd.Series("", index=df.index)
    for f in fields:
        if f in df.columns:
            b = b + " " + df[f].fillna("").astype(str)
    return b.str.lower()


def count_terms(df, mapping, fields=("Title", "Abstract", "Keyword")):
    """Records matching each category. A record can match several, so shares are
    computed over total matches rather than over records."""
    b = text_blob(df, fields)
    return pd.Series(
        {k: int(b.str.contains("|".join(re.escape(t) for t in terms), regex=True).sum())
         for k, terms in mapping.items()}).sort_values(ascending=False)


def classify_best(df, mapping, fields=("Title", "Abstract", "Keyword"), other="Other"):
    """Assign each record to the category with the most term hits; ties by order."""
    b = text_blob(df, fields)
    scores = pd.DataFrame(
        {k: sum(b.str.count(re.escape(t)) for t in terms) for k, terms in mapping.items()})
    best = scores.idxmax(axis=1)
    best[scores.max(axis=1) == 0] = other
    return best


print("Category definitions loaded. Edit the dictionaries above to re-scope the review.")

---
## B1 — Figure 3: the four bibliometric panels

**a) Keyword distribution** — which of the five extraction keywords each record
matches. The paper found AI dominant at roughly 69% with over 600 instances, and
concluded that AI is studied as a broad concept while its practical application to
decision-making and policymaking is much thinner.

**b) Yearly distribution** — the publication trend. The paper reports a slow period
1960–2000, a rise from 2000, a spike after 2010 and a peak around 2020.

**c) Study types** — requires a `StudyType` column. If your corpus has none, the panel
says so rather than inventing one; add the column during full-text screening.

**d) Journal distribution** — the top journals by record count.

In [ ]:
keyword_counts = count_terms(corpus, EXTRACTION_KEYWORDS)
yearly = corpus["Year"].value_counts().sort_index()
full_years = pd.Series(0, index=pd.RangeIndex(int(corpus["Year"].min()),
                                              int(corpus["Year"].max()) + 1, name="Year"))
yearly = full_years.add(yearly, fill_value=0)

has_types = "StudyType" in corpus.columns and corpus["StudyType"].notna().any()
study_types = (corpus["StudyType"].dropna().value_counts()
               if has_types else pd.Series(dtype=int))
journals = corpus["Journal"].replace("", np.nan).dropna().value_counts()

SHORT_JOURNAL = {"Journal of The American Planning Association": "JAPA",
                 "Journal of Planning Education and Research": "JPER",
                 "Journal of the Operational Research Society": "JORS",
                 "Land Use Policy": "LUP", "Science and Engineering Ethics": "SEE",
                 "Computers Environment and Urban Systems": "CEUS",
                 "Sustainable Cities and Society": "SCS",
                 "Government Information Quarterly": "GIQ"}

fig, axs = plt.subplots(2, 2, figsize=(16, 11))

ax = axs[0, 0]
share = keyword_counts / keyword_counts.sum() * 100
wedges, _, auto = ax.pie(share.values, labels=None, autopct="%1.0f%%",
                         startangle=90, colors=PALETTE[::-1][:len(share)],
                         wedgeprops=dict(width=0.42, edgecolor="white"),
                         pctdistance=0.78, textprops=dict(fontsize=8))
ax.legend(wedges, [f"{k} ({v:,})" for k, v in keyword_counts.items()],
          fontsize=7, loc="center left", bbox_to_anchor=(0.97, 0.5))
ax.set_title("a) Keyword distribution")

ax = axs[0, 1]
ax.fill_between(yearly.index, yearly.values, color=PALETTE[0], alpha=0.75)
ax.plot(yearly.index, yearly.values, color=PALETTE[3], lw=1.6)
ax.set(title="b) Yearly distribution", xlabel="Year", ylabel="No. of studies")
peak = int(yearly.idxmax())
ax.annotate(f"peak {peak}", xy=(peak, yearly.max()),
            xytext=(-46, -18), textcoords="offset points", fontsize=8,
            arrowprops=dict(arrowstyle="->", color="grey", lw=0.9))

ax = axs[1, 0]
if has_types:
    st = study_types.head(8)[::-1]
    ax.barh(range(len(st)), st.values, color=PALETTE[2], height=0.65)
    ax.set_yticks(range(len(st))); ax.set_yticklabels(st.index, fontsize=8)
    for i, v in enumerate(st.values):
        ax.text(v, i, f" {v:,}", va="center", fontsize=8)
    ax.set(title="c) Study-type distribution", xlabel="No. of studies")
else:
    ax.text(0.5, 0.5, "No StudyType column in the corpus.\n\n"
                      "Add one during full-text screening\nand re-run this cell.",
            ha="center", va="center", fontsize=10, color="grey")
    ax.set(title="c) Study-type distribution"); ax.axis("off")

ax = axs[1, 1]
top_j = journals.head(6)
labels = [SHORT_JOURNAL.get(j, j)[:26] for j in top_j.index]
other = journals.sum() - top_j.sum()
vals = list(top_j.values) + ([other] if other > 0 else [])
labels = labels + (["Other"] if other > 0 else [])
ax.pie(vals, labels=labels, autopct="%1.0f%%", startangle=140,
       colors=(PALETTE * 3)[:len(vals)], textprops=dict(fontsize=7.5))
ax.set_title("d) Journal distribution")

fig.suptitle("Figure 3 — Bibliometric and trend analysis", fontsize=13, weight="bold")
fig.tight_layout()
finish(fig, "figure3_bibliometric")

for name, obj in [("keyword_counts", keyword_counts), ("yearly_publications", yearly),
                  ("study_types", study_types), ("journal_counts", journals)]:
    obj.to_frame("count").to_csv(TAB_DIR / f"{name}.csv")
print(f"  tables -> {TAB_DIR}")

In [ ]:
# ---- How close is your corpus to the published values? ----
def compare(observed, published, label):
    obs_share = (observed / observed.sum() * 100).round(1)
    pub = pd.Series(published, dtype=float)
    pub_share = (pub / pub.sum() * 100).round(1)
    out = pd.DataFrame({"yours_%": obs_share, "paper_%": pub_share}).fillna(0)
    out["diff"] = (out["yours_%"] - out["paper_%"]).round(1)
    print(f"\n{label}")
    display(out.sort_values("paper_%", ascending=False))
    return out


print("=" * 62)
print(f"Corpus size: {len(corpus):,}   Paper: 588")
print("=" * 62)
compare(keyword_counts, PAPER_KEYWORD_SHARE, "Keyword distribution")
if has_types:
    compare(study_types, PAPER_STUDY_TYPES, "Study types")
if IS_DEMO:
    print("\n  Demo corpus - large differences are expected and mean nothing.")

---
## B2 — Figure 5a–b: application areas and disciplines

Section 3.2.1 classifies each record into an application area and an academic
discipline by parsing title, abstract and journal. Each record is assigned to the
category with the most term hits; anything with no hits becomes `Other`, which is
reported rather than dropped so you can see how much the classifier is missing.

Journal name **is** used here, unlike in the geographic analysis in notebook 03 —
the journal is legitimate evidence of a paper's discipline, but not of where its
study was carried out.

In [ ]:
corpus["ApplicationArea"] = classify_best(corpus, APPLICATION_AREAS)
corpus["Discipline"] = classify_best(
    corpus, DISCIPLINES, fields=("Title", "Abstract", "Keyword", "Journal"))

app_counts = corpus["ApplicationArea"].value_counts()
disc_counts = corpus["Discipline"].value_counts()

fig, axs = plt.subplots(2, 1, figsize=(12, 11))

for ax, counts, paper, title, letter in [
        (axs[0], app_counts, PAPER_APPLICATION_COUNTS, "Application area", "a"),
        (axs[1], disc_counts, PAPER_DISCIPLINE_COUNTS, "Discipline", "b")]:
    order = [k for k in paper if k in counts.index] + \
            [k for k in counts.index if k not in paper]
    vals = counts.reindex(order).fillna(0)[::-1]
    ax.barh(range(len(vals)), vals.values, color=PALETTE[2], height=0.62)
    ax.set_yticks(range(len(vals)))
    ax.set_yticklabels([str(i)[:46] for i in vals.index], fontsize=8)
    for i, v in enumerate(vals.values):
        ax.text(v, i, f"  {int(v):,}", va="center", fontsize=8)
    ax.set(title=f"{letter}) {title}", xlabel="Number of papers")

fig.suptitle("Figure 5a–b — Application areas and disciplines", fontsize=13, weight="bold")
fig.tight_layout()
finish(fig, "figure5ab_areas_disciplines")

unclassified = int((corpus["ApplicationArea"] == "Other").sum())
print(f"\nUnclassified by application area: {unclassified:,} "
      f"({unclassified / len(corpus):.0%})")
print("  If that share is high, widen the term lists in APPLICATION_AREAS above")
print("  rather than accepting the classifier's silence as a finding.\n")

compare(app_counts, PAPER_APPLICATION_COUNTS, "Application areas")
compare(disc_counts, PAPER_DISCIPLINE_COUNTS, "Disciplines")

corpus.to_csv(DATA_DIR / "corpus_classified.csv", index=False)
print(f"\nWrote corpus_classified.csv (adds ApplicationArea and Discipline)")
print("Notebooks 03 and 04 will pick this up automatically.")

---

Next: **`03_geographic_mapping.ipynb`**, then **`04_network_and_matrix.ipynb`**.